# Gold Layer

This notebook creates business-ready reports from the cleaned Silver dataset.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "hrm6131_landing"

SILVER_TICKETS = f"{CATALOG}.{SCHEMA}.silver_tickets"

GOLD_TEAM_PERFORMANCE = f"{CATALOG}.{SCHEMA}.gold_team_performance"
GOLD_AGENT_PERFORMANCE = f"{CATALOG}.{SCHEMA}.gold_agent_performance"
GOLD_QUALITY_COMPLIANCE = f"{CATALOG}.{SCHEMA}.gold_quality_compliance"
GOLD_DAY2_CARRYOVER = f"{CATALOG}.{SCHEMA}.gold_day2_carryover"

## Load Silver Data

In [0]:
silver = spark.table(SILVER_TICKETS)

print("Silver Records:", silver.count())

display(silver)

Silver Records: 134


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes
A001,TKT00001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,35,45,36
A001,TKT00002,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,22,10,22
A022,TKT00061,Resolved,0h 44m 0s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A022,Support Agent,TL05,0,44,0,44
A025,TKT00069,Resolved,0h 26m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A025,Senior Support Agent,TL05,0,26,0,26
A033,TKT00094,Resolved,0h 19m 45s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A033,Junior Support Agent,TL07,0,19,45,20
A004,TKT00011,Resolved,0h 33m 15s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A004,Junior Support Agent,TL01,0,33,15,33
A016,TKT00045,Resolved,0h 36m 0s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A016,Junior Support Agent,TL04,0,36,0,36
A026,TKT00072,Resolved,0h 23m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A026,Support Agent,TL06,0,23,0,23
A031,TKT00086,Resolved,0h 29m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A031,Senior Support Agent,TL07,0,29,0,29
A034,TKT00096,Resolved,0h 25m 15s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A034,Support Agent,TL07,0,25,15,25


## Team Performance Report

In [0]:
team_performance = (
    silver
    .groupBy(
        "day",
        "team_lead_id"
    )
    .agg(
        F.count("ticket_id").alias("tickets_resolved"),
        F.round(
            F.avg("resolution_time_minutes"),
            2
        ).alias("avg_resolution_time")
    )
    .orderBy(
        "day",
        "team_lead_id"
    )
)

display(team_performance)

day,team_lead_id,tickets_resolved,avg_resolution_time
1,TL01,10,30.9
1,TL02,10,31.4
1,TL03,10,32.0
1,TL04,9,26.56
1,TL05,10,31.6
1,TL06,10,29.4
1,TL07,10,27.8
1,TL08,4,25.25
2,TL01,7,25.14
2,TL02,8,26.25


## Save Team Performance Report

In [0]:
(
    team_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TEAM_PERFORMANCE)
)

print("Team Performance report saved successfully.")

Team Performance report saved successfully.


## Validate Team Performance Report

In [0]:
team = spark.table(GOLD_TEAM_PERFORMANCE)

print("Total Teams:", team.count())

display(team)

Total Teams: 16


day,team_lead_id,tickets_resolved,avg_resolution_time
1,TL01,10,30.9
1,TL02,10,31.4
1,TL03,10,32.0
1,TL04,9,26.56
1,TL05,10,31.6
1,TL06,10,29.4
1,TL07,10,27.8
1,TL08,4,25.25
2,TL01,7,25.14
2,TL02,8,26.25


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS default.customer_support_pipeline
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.customer_support_pipeline
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW VOLUMES IN workspace.default
""").display()

database,volume_name
default,customer_support_pipeline
default,week5_assignment
default,week6_files
default,week7_files


In [0]:
(
    team
    .coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header","true")
    .save("/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance")
)

print("Team Performance CSV exported successfully.")

Team Performance CSV exported successfully.


In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/_SUCCESS,_SUCCESS,0,1784404517000
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/_committed_1441189778292849864,_committed_1441189778292849864,212,1784404517000
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/_committed_5196199348510470301,_committed_5196199348510470301,113,1784402002000
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/_committed_vacuum6839697894344733881,_committed_vacuum6839697894344733881,96,1784404517000
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/_started_1441189778292849864,_started_1441189778292849864,0,1784404516000
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/part-00000-tid-1441189778292849864-cd1cbbfb-8f8a-4b0f-988e-eadd0f3be612-861-1-c000.csv,part-00000-tid-1441189778292849864-cd1cbbfb-8f8a-4b0f-988e-eadd0f3be612-861-1-c000.csv,293,1784404516000


In [0]:
team.toPandas().to_csv(
    "/Workspace/Customer_Support_Ticket_Resolution_Pipeline/outputs/team_performance.csv",
    index=False
)

print("Team Performance copied to project outputs folder")

Team Performance copied to project outputs folder


## Agent Performance Report

In [0]:
agent_performance = (
    silver
    .groupBy(
        "day",
        "agent_id",
        "agent_name",
        "team_lead_id"
    )
    .agg(
        F.count("ticket_id").alias("tickets_resolved"),
        F.round(
            F.avg("resolution_time_minutes"),
            2
        ).alias("avg_resolution_time")
    )
    .orderBy(
        "day",
        "team_lead_id",
        "agent_id"
    )
)

display(agent_performance)

day,agent_id,agent_name,team_lead_id,tickets_resolved,avg_resolution_time
1,A001,Agent_A001,TL01,3,26.0
1,A002,Agent_A002,TL01,2,43.5
1,A003,Agent_A003,TL01,2,22.5
1,A004,Agent_A004,TL01,2,27.0
1,A005,Agent_A005,TL01,1,45.0
1,A006,Agent_A006,TL02,2,23.0
1,A007,Agent_A007,TL02,2,44.0
1,A008,Agent_A008,TL02,2,25.0
1,A009,Agent_A009,TL02,2,30.5
1,A010,Agent_A010,TL02,2,34.5


## Save Agent Performance Report

In [0]:
(
    agent_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_AGENT_PERFORMANCE)
)

print("Agent Performance report saved successfully.")

Agent Performance report saved successfully.


## Validate Agent Performance Report

In [0]:
agent = spark.table(GOLD_AGENT_PERFORMANCE)

print("Total Records:", agent.count())

display(agent)

Total Records: 77


day,agent_id,agent_name,team_lead_id,tickets_resolved,avg_resolution_time
1,A001,Agent_A001,TL01,3,26.0
1,A002,Agent_A002,TL01,2,43.5
1,A003,Agent_A003,TL01,2,22.5
1,A004,Agent_A004,TL01,2,27.0
1,A005,Agent_A005,TL01,1,45.0
1,A006,Agent_A006,TL02,2,23.0
1,A007,Agent_A007,TL02,2,44.0
1,A008,Agent_A008,TL02,2,25.0
1,A009,Agent_A009,TL02,2,30.5
1,A010,Agent_A010,TL02,2,34.5


In [0]:
(
    agent
    .coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header","true")
    .save("/Volumes/workspace/default/customer_support_pipeline/outputs/agent_performance")
)

print("Agent Performance CSV exported successfully.")

Agent Performance CSV exported successfully.


In [0]:
agent.toPandas().to_csv(
    "/Workspace/Customer_Support_Ticket_Resolution_Pipeline/outputs/agent_performance.csv",
    index=False
)

print("Agent Performance copied")

Agent Performance copied


## Quality Compliance Report

In [0]:
quality_compliance = (
    silver
    .groupBy(
        "team_lead_id",
        "agent_id",
        "agent_name"
    )
    .agg(
        F.count("ticket_id").alias("quality_compliant_tickets")
    )
    .orderBy(
        "team_lead_id",
        F.desc("quality_compliant_tickets")
    )
)

display(quality_compliance)

team_lead_id,agent_id,agent_name,quality_compliant_tickets
TL01,A001,Agent_A001,5
TL01,A004,Agent_A004,4
TL01,A003,Agent_A003,3
TL01,A002,Agent_A002,3
TL01,A005,Agent_A005,2
TL02,A010,Agent_A010,4
TL02,A009,Agent_A009,4
TL02,A007,Agent_A007,4
TL02,A006,Agent_A006,3
TL02,A008,Agent_A008,3


In [0]:
(
    quality_compliance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_QUALITY_COMPLIANCE)
)

print("Quality Compliance report saved successfully.")

Quality Compliance report saved successfully.


In [0]:
quality = spark.table(GOLD_QUALITY_COMPLIANCE)

print("Total Records:", quality.count())

display(quality)

Total Records: 40


team_lead_id,agent_id,agent_name,quality_compliant_tickets
TL01,A001,Agent_A001,5
TL01,A004,Agent_A004,4
TL01,A002,Agent_A002,3
TL01,A003,Agent_A003,3
TL01,A005,Agent_A005,2
TL02,A009,Agent_A009,4
TL02,A010,Agent_A010,4
TL02,A007,Agent_A007,4
TL02,A006,Agent_A006,3
TL02,A008,Agent_A008,3


In [0]:
(
    quality
    .coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header","true")
    .save("/Volumes/workspace/default/customer_support_pipeline/outputs/quality_compliance")
)

print("Quality Compliance CSV exported successfully.")

Quality Compliance CSV exported successfully.


In [0]:
quality.toPandas().to_csv(
    "/Workspace/Customer_Support_Ticket_Resolution_Pipeline/outputs/quality_compliance.csv",
    index=False
)

print("Quality Compliance copied")

Quality Compliance copied


## Day 2 Carry-over Report

In [0]:
day1_success_agents = (
    silver
    .filter(F.col("day") == 1)
    .select("agent_id")
    .distinct()
)

print("Successful Day 1 Agents:", day1_success_agents.count())

display(day1_success_agents)

Successful Day 1 Agents: 37


agent_id
A001
A022
A025
A033
A004
A016
A026
A031
A034
A005


In [0]:
(
    day2_carryover.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DAY2_CARRYOVER)
)

print("Day 2 Carry-over report saved successfully.")

Day 2 Carry-over report saved successfully.


In [0]:
carryover = spark.table(GOLD_DAY2_CARRYOVER)

print("Total Carry-over Records:", carryover.count())

display(carryover)

Total Carry-over Records: 4


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes
A040,TKT00278,Resolved,0h 22m 0s,Technical,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A040,Junior Support Agent,TL08,0,22,0,22
A039,TKT00276,Resolved,0h 35m 45s,Returns,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A039,Support Agent,TL08,0,35,45,36
A040,TKT00279,Resolved,0h 29m 30s,Billing,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A040,Junior Support Agent,TL08,0,29,30,30
A038,TKT00274,Resolved,0h 26m 0s,Account,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A038,Senior Support Agent,TL08,0,26,0,26


In [0]:
print("Unique Day 1 Agents:")
display(
    silver.filter(F.col("day") == 1)
          .select("agent_id")
          .distinct()
)

Unique Day 1 Agents:


agent_id
A001
A022
A025
A033
A004
A016
A026
A031
A034
A005


In [0]:
print("Unique Day 2 Agents:")
display(
    silver.filter(F.col("day") == 2)
          .select("agent_id")
          .distinct()
)

Unique Day 2 Agents:


agent_id
A001
A022
A025
A033
A004
A016
A026
A031
A034
A005


In [0]:
print("Agents in Day 1:", silver.filter(F.col("day") == 1).select("agent_id").distinct().count())
print("Agents in Day 2:", silver.filter(F.col("day") == 2).select("agent_id").distinct().count())

Agents in Day 1: 37
Agents in Day 2: 40


In [0]:
print("Team Report:", team.count())
print("Agent Report:", agent.count())
print("Quality Report:", quality.count())
print("Carry-over:", carryover.count())

Team Report: 16
Agent Report: 77
Quality Report: 40
Carry-over: 4


In [0]:
(
    carryover
    .coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header","true")
    .save("/Volumes/workspace/default/customer_support_pipeline/outputs/day2_carryover")
)

print("Day2 Carry-over CSV exported successfully.")

Day2 Carry-over CSV exported successfully.


In [0]:
carryover.toPandas().to_csv(
    "/Workspace/Customer_Support_Ticket_Resolution_Pipeline/outputs/day2_carryover.csv",
    index=False
)

print("Carry-over copied")

Carry-over copied


In [0]:
display(
dbutils.fs.ls(
"/Volumes/workspace/default/customer_support_pipeline/outputs"
))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/agent_performance/,agent_performance/,0,1784404872817
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/day2_carryover/,day2_carryover/,0,1784404872817
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/quality_compliance/,quality_compliance/,0,1784404872818
dbfs:/Volumes/workspace/default/customer_support_pipeline/outputs/team_performance/,team_performance/,0,1784404872818
